# NumPy Basics and Complete Linear Regression
## Project: Employee Productivity Score Predictor (9 features)

**Which features affect the productivity of an employee?**

This notebook goes with the 2-hour teaching guide. It puts all the code in one place:

* **Part A - NumPy basics** (arrays, picking items, masks, maths on arrays, broadcasting, matrices, random numbers)
* **Part B - Load the data, baseline, correlations**
* **Part C to H - Linear regression, built only with scikit-learn** (`LinearRegression`, `SGDRegressor`, `StandardScaler`, `train_test_split`, `Ridge`, metrics). NumPy is used only to hold and prepare the data (arrays, masks, plots), not to build any regression model.

> **Read this first: the data is made up.** `employee_productivity_data.csv` is simulated: 250 employees x 24 months = 6,000 rows, one row per employee-month. The patterns in it were put in on purpose. Never present these results as facts about real employees, and never use a predicted score to judge a person.

**The 9 features:** `experience_years`, `training_hours`, `task_complexity`, `ai_assistant`, `rework_tickets`, `absent_days`, `team_size`, `meeting_hours`, `blocker_hours`.
**Target:** `productivity_score` = tasks completed / 160 x 100, capped at 100. (`tasks_completed` is never used as a feature, because it would give away the answer.)

## 0. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# scikit-learn tools used for ALL the regression work
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, SGDRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (mean_squared_error, root_mean_squared_error,
                             mean_absolute_error, r2_score)

np.set_printoptions(suppress=True, linewidth=120)
print("NumPy", np.__version__)

In [ ]:
# Get the data file. In Colab: run this cell and upload employee_productivity_data.csv
CSV = "employee_productivity_data.csv"
if not os.path.exists(CSV):
    try:
        from google.colab import files
        print("Please choose employee_productivity_data.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put employee_productivity_data.csv next to this notebook.")
print("File found:", os.path.exists(CSV))

---
# Part A - NumPy basics
We use six real rows from the data (the first six employee-months) so you can check every answer by eye.

### A1. Why NumPy: speed

In [ ]:
data_list = list(range(1_000_000))
data_arr  = np.arange(1_000_000)

timeit [x * 2 for x in data_list]   # list
timeit data_arr * 2                 # numpy

### A2. Creating arrays and checking their properties

In [ ]:
exp6    = np.array([4, 3.4, 3, 4.1, 2.9, 5.3])    # experience_years
rework6 = np.array([3, 5, 5, 0, 1, 4])             # rework_tickets
absent6 = np.array([1, 1, 1, 1, 0, 1])             # absent_days
score6  = np.array([56.9, 50.6, 67.5, 65, 70, 56.2])   # productivity_score

print(np.zeros((2, 3)))
print(np.ones(4))
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))
print(np.eye(3))

records = np.array([[4, 3, 56.9],
                    [3.4, 5, 50.6],
                    [3, 5, 67.5]])      # rows = employee-months, columns = measurements
print("ndim :", records.ndim)
print("shape:", records.shape)          # (rows, columns)
print("size :", records.size)
print("dtype:", records.dtype)

**Poll 1:** what is the shape of `np.zeros((4, 3))`?

In [ ]:
print(np.zeros((4, 3)).shape)   # (4, 3): rows first, then columns

### A3. Indexing and slicing (positions start at 0)

In [ ]:
print(exp6[0], exp6[-1])        # first and last
print(exp6[1:4])                # start inclusive, stop exclusive
print(exp6[::2])                # every second record
print(records[:, 0])            # all rows, column 0
print(records[2, 1])            # row 2, column 1
print(records[1:3, :])          # rows 1 and 2, all columns

# A slice is a WINDOW onto the original, not a copy
s = exp6[:3]
s[0] = 999
print("original changed:", exp6[0])
exp6[0] = 4                     # restore it
safe = exp6[:3].copy()          # an independent copy

### A4. Filtering with a boolean mask

In [ ]:
print(exp6 > 3)                                  # a mask of True/False
print(score6[exp6 > 3])                          # scores of people with more than 3 years
good = (exp6 > 3) & (rework6 <= 3)               # use & | ~ with brackets
print(score6[good])
print(np.where(score6 >= 60, "strong month", "weaker month"))

values = np.array([88.0, np.nan, 91.0])          # NaN = a missing value
print(np.isnan(values))
print(values[~np.isnan(values)])                 # real numbers only

**Poll 2:** scores of people with more than 3 and fewer than 5 years of experience. Option B is right: `& ` with brackets.

In [ ]:
print(score6[(exp6 > 3) & (exp6 < 5)])   # [56.9 50.6 65. ]

### A5. Maths on whole arrays (vectorisation) and the simple guess

In [ ]:
avg   = score6.mean()               # 61.0
guess = np.full(6, avg)             # the same guess for everyone
error = score6 - guess              # real minus guess
gap   = 100 - score6                # distance from the maximum score
print(avg, error.round(1), gap.round(1), sep="\n")

### A6. Summaries and the `axis` argument

In [ ]:
print("mean  :", score6.mean())
print("std   :", score6.std().round(1))
print("argmax:", score6.argmax(), "(the POSITION of the highest score)")
print("median:", np.median(score6))
print("cumsum:", np.cumsum(score6))
print(records.mean(axis=0))         # one value per COLUMN (squashes the rows)
print(records.mean(axis=1))         # one value per ROW (squashes the columns)

### A7. Broadcasting and standardising

In [ ]:
print(records - records.mean(axis=0))      # (3,3) - (3,): subtract each column's mean

# Standardising: average 0, spread 1
z = (exp6 - exp6.mean()) / exp6.std()
print(z.round(2), z.mean().round(6), z.std().round(6))

### A8. Reshaping, joining, matrix maths and random numbers

In [ ]:
a = np.arange(6)
print(a.reshape(2, 3))
print(a.reshape(-1, 1).shape)            # column vector; -1 means "work it out"
print(a.reshape(2, 3).T.shape)           # transpose
print(np.column_stack([exp6, rework6, absent6]).shape)   # features side by side

A = np.array([[1, 2], [3, 4]]); v = np.array([5, 6])
print(A * A)                 # element-wise product
print(A @ A)                 # MATRIX product
print(A @ v)                 # [17 39]
print(np.linalg.solve(A, v)) # solves A x = v

rng = np.random.default_rng(42)          # the seed makes results repeatable
print(rng.integers(20, 80, size=5))
print(rng.normal(0, 8, size=5).round(2))
print(rng.permutation(10))

---
# Part B - Load the data, remove blanks, baseline, correlations
`np.genfromtxt` turns blank cells into `NaN` (only `training_hours` has blanks: 162 of them).

In [ ]:
data = np.genfromtxt(CSV, delimiter=",", skip_header=1)
month    = data[:, 2]
features = data[:, 3:12]      # the 9 feature columns
score    = data[:, 13]        # the target (productivity_score)

names = ["experience_years", "training_hours", "task_complexity",
         "ai_assistant", "rework_tickets", "absent_days", "team_size",
         "meeting_hours", "blocker_hours"]

print(data.shape)                         # (6000, 14)
print(np.isnan(features).sum(axis=0))     # blanks: [0 162 0 0 0 0 0 0 0]

ok = ~np.isnan(features).any(axis=1)      # True for rows with no blank
F, y, mo = features[ok], score[ok], month[ok]
print(F.shape)                            # (5838, 9)
print("average score:", y.mean().round(1), "| average experience:", F[:, 0].mean().round(1))

### B1. Which features move together with the score? (correlation)

In [ ]:
cors = np.array([np.corrcoef(F[:, j], y)[0, 1] for j in range(9)])
order = np.argsort(-np.abs(cors))
for j in order:
    print(f"{names[j]:18s} {cors[j]:+.2f}")

plt.figure(figsize=(6, 3.2))
plt.barh([names[j] for j in order][::-1], cors[order][::-1], color="steelblue")
plt.axvline(0, color="black", lw=0.8); plt.title("Correlation of each feature with the score")
plt.tight_layout(); plt.show()

A correlation shows a link. It does **not** prove that one thing causes the other.

### B2. The baseline: predict the average for everyone

In [ ]:
def rmse(a, b):
    return root_mean_squared_error(a, b)      # square root of the mean squared error

# DummyRegressor(strategy="mean") is scikit-learn's "predict the average for everyone"
baseline = DummyRegressor(strategy="mean").fit(F, y)
base_pred = baseline.predict(F)
print("Baseline RMSE:", round(rmse(y, base_pred), 1))        # about 9.1

---
# Part C - Simple regression: one feature (experience)
Line: `score = m * experience + c`. `LinearRegression` finds `m` and `c` by making the mean squared error (MSE) as small as possible.

### C1. Fit the line with scikit-learn

In [ ]:
x = F[:, [0]]                      # experience_years as a 2D column: (rows, 1). sklearn wants X in 2D
lr1 = LinearRegression()
lr1.fit(x, y)
p1 = lr1.predict(x)

print("slope m:", lr1.coef_[0].round(2), "| intercept c:", round(lr1.intercept_, 1))     # about 1.88 and 53.2
print("RMSE", round(rmse(y, p1), 1),
      "| MAE", round(mean_absolute_error(y, p1), 1),
      "| R2", round(r2_score(y, p1), 3))                                                 # 8.0, 6.4, 0.231
print("Predicted score for 6 years of experience:", lr1.predict([[6]])[0].round(1))

### C2. Draw the fitted line

In [ ]:
plt.figure(figsize=(6, 3.6))
sample = np.random.default_rng(1).choice(len(y), 600, replace=False)
plt.scatter(x[sample, 0], y[sample], s=8, alpha=0.4, label="employee-months (600 shown)")
xs = np.linspace(x.min(), x.max(), 50)
plt.plot(xs, lr1.predict(xs.reshape(-1, 1)), color="red", label="fitted line")
plt.xlabel("experience_years"); plt.ylabel("productivity_score"); plt.legend(); plt.tight_layout(); plt.show()

### C3. Each feature on its own (R-squared with one feature at a time)

In [ ]:
rows = []
for j in range(9):
    mdl = LinearRegression().fit(F[:, [j]], y)
    pj = mdl.predict(F[:, [j]])
    rows.append((names[j], mdl.coef_[0], np.sqrt(mean_squared_error(y, pj)), r2_score(y, pj)))
rows.sort(key=lambda r: -r[3])
print(f"{'feature':18s} {'slope':>7s} {'RMSE':>6s} {'R2':>6s}")
for nme, sl, rm, r2_ in rows:
    print(f"{nme:18s} {sl:+7.2f} {rm:6.1f} {r2_:6.3f}")

### C4. Debug Challenge: Find and Fix the Bug
**Scenario:** A student attempts to fit a simple linear regression using `training_hours` (column index 1 in feature matrix `F`) to predict `productivity_score` (`y`). When calling `.fit()`, scikit-learn crashes with an error.

> `ValueError: Expected 2D array, got 1D array instead`

**Your Task:**
1. Run the cell below to inspect the traceback and error message.
1. Identify why scikit-learn rejected `X_training`.
1. Fix the slicing syntax so `X_training` is passed as a 2D matrix with shape `(n_samples, 1)`.

<details>
<summary>💡 Hint</summary>
<code>F[:, 1]</code> produces a 1D vector of shape <code>(N,)</code>. Scikit-learn models require a 2D array of shape <code>(n_samples, n_features)</code>. Use <code>F[:, [1]]</code> or <code>X_training.reshape(-1, 1)</code>.
</details>

In [ ]:
# ❌ BUGGY CODE: Run this cell to observe the error, then fix it!
# Goal: Fit a simple linear regression using training_hours to predict productivity_score.
# Feature column 1 is training_hours.

X_training = F[:, 1]  # 💥 Bug: 1D array slice of shape (N,)
y_target = y

debug_lr = LinearRegression()
# This line crashes because X is 1D:
debug_lr.fit(X_training, y_target)

print("Slope m:", round(debug_lr.coef_[0], 3))
print("Intercept c:", round(debug_lr.intercept_, 3))

---
# Part D - Gradient descent and the learning rate
Start with a guess, find which way is downhill, take a small step, repeat. scikit-learn does this in `SGDRegressor`. We standardise `x` first with `StandardScaler` so the weight and the intercept learn at a similar speed.

### D1. Gradient descent with `SGDRegressor`

In [ ]:
sgd = make_pipeline(StandardScaler(),
                    SGDRegressor(loss="squared_error", penalty=None, learning_rate="constant",
                                 eta0=0.01, max_iter=2000, tol=None, random_state=0))
sgd.fit(x, y)
psgd = sgd.predict(x)
print("SGD RMSE:", round(rmse(y, psgd), 2), "| LinearRegression RMSE (the best possible):", round(rmse(y, p1), 2))
print("Gap to the best possible RMSE:", round(rmse(y, psgd) - rmse(y, p1), 3), "(tiny -> gradient descent found almost the same line)")

### D2. The learning-rate experiment

In [ ]:
import warnings

def sgd_losses(eta0, epochs=15):
    # partial_fit does ONE pass over the data each time, so we can watch the loss fall epoch by epoch
    xs_ = StandardScaler().fit_transform(x)
    reg = SGDRegressor(loss="squared_error", penalty=None, learning_rate="constant",
                       eta0=eta0, random_state=0)
    losses = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for _ in range(epochs):
            try:
                reg.partial_fit(xs_, y)
                losses.append(mean_squared_error(y, reg.predict(xs_)))
            except ValueError:                 # the numbers blew up (overflow)
                losses.append(float("inf"))
    return losses

plt.figure(figsize=(6, 3.4))
for eta, label in [(0.00001, "0.00001 (too small)"), (0.01, "0.01 (just right)"), (1.0, "1.0 (too big)")]:
    L = sgd_losses(eta)
    print(f"eta0={eta:<8} final loss = {L[-1]:.3g}")
    plt.plot(range(1, len(L) + 1), [min(v, 1e5) for v in L], label=label)
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("MSE (clipped)"); plt.legend(); plt.tight_layout(); plt.show()

### D3. Debug Challenge: Find and Fix the Bug
**Scenario:** A student sets up an `SGDRegressor` pipeline with a constant learning rate to predict productivity scores from standardised experience. However, gradient descent diverges and produces an overflow or `NaN` predictions!

> `ValueError: Floating point issues in SGDRegressor (loss overflow or infinite values)`

**Your Task:**
1. Run the cell below to observe the gradient descent explosion.
1. Look closely at the hyperparameters passed to `SGDRegressor`. Which parameter controls step size?
1. Fix the learning rate `eta0` to a sensible value so gradient descent converges to a finite, accurate RMSE.

<details>
<summary>💡 Hint</summary>
A learning rate of <code>eta0=5.0</code> is huge! It causes each step to wildly overshoot the loss minimum and blow up. Try a standard learning rate like <code>0.01</code> or <code>0.001</code>.
</details>

In [ ]:
# ❌ BUGGY CODE: Run this cell to observe the error, then fix it!
# Goal: Train SGDRegressor with constant learning rate to predict score from standardised experience.
# Why does gradient descent diverge and blow up?

sgd_broken = make_pipeline(
    StandardScaler(),
    SGDRegressor(loss="squared_error", penalty=None, learning_rate="constant",
                 eta0=5.0, max_iter=1000, random_state=42)  # 💥 Look at eta0!
)

# Run this to observe the divergence / overflow:
sgd_broken.fit(x, y)
print("Sample predictions:", sgd_broken.predict(x[:3]))
print("RMSE:", round(rmse(y, sgd_broken.predict(x)), 2))

---
# Part E - All nine features together
### E1. Fit all nine features with `LinearRegression`

In [ ]:
model = LinearRegression().fit(F, y)             # sklearn adds the intercept for us
print("intercept:", round(model.intercept_, 2))
for nme, w in zip(names, model.coef_):
    print(f"  {nme:18s} {w:+.2f}")

pm = model.predict(F)
r2_all = r2_score(y, pm)
k = 9                                            # number of features
adj_r2 = 1 - (1 - r2_all) * (len(y) - 1) / (len(y) - k - 1)
print("RMSE", round(rmse(y, pm), 1), "| MAE", round(mean_absolute_error(y, pm), 1),
      "| R2", round(r2_all, 3), "| adjusted R2", round(adj_r2, 3))          # 4.5, 3.6, 0.755, 0.754

# Two example people
#  6 yrs, 10 h training, complexity 3, AI yes, 2 rework, 1 absent, team 8, 14 meeting h, 6 blocker h
person1 = np.array([[6, 10, 3, 1, 2, 1, 8, 14, 6]])
#  2 yrs, 4 h training, complexity 4, no AI, 5 rework, 3 absent, team 6, 20 meeting h, 14 blocker h
person2 = np.array([[2, 4, 4, 0, 5, 3, 6, 20, 14]])
print("Person 1:", model.predict(person1)[0].round(1), "| Person 2:", model.predict(person2)[0].round(1))

What does each weight mean? Holding the other features the same, one more unit of that feature changes the predicted score by the weight. For `ai_assistant` (yes/no) the weight is the gap between yes and no.

### E2. Gradient descent with many features (scaling matters)

In [ ]:
sgd_params = dict(loss="squared_error", penalty=None, learning_rate="constant",
                  eta0=0.01, max_iter=300, tol=None, random_state=0)

# 1) With StandardScaler: all nine features are on a similar scale, so one learning rate suits all
sgd_scaled = make_pipeline(StandardScaler(), SGDRegressor(**sgd_params)).fit(F, y)
print("Scaled   -> RMSE:", round(rmse(y, sgd_scaled.predict(F)), 2),
      "| LinearRegression RMSE:", round(rmse(y, pm), 2))

# 2) Without scaling: meeting_hours (about 16) and blocker_hours (about 10) are big numbers, so the steps overshoot
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:
        sgd_raw = SGDRegressor(**sgd_params).fit(F, y)
        print("Unscaled -> RMSE:", round(rmse(y, sgd_raw.predict(F)), 2))
    except ValueError:
        print("Unscaled -> the numbers blew up (overflow). Same learning rate, no scaling = failure.")

# The scaler remembers each feature's average and spread (we need these in Part F)
scaler = StandardScaler().fit(F)
mu, sd = scaler.mean_, scaler.scale_
print("average of each feature:", mu.round(1))
print("spread of each feature :", sd.round(1))

### E3. Do the two new features earn their place? (7 features vs 9 features)

In [ ]:
def fit_report(cols):
    m_ = LinearRegression().fit(F[:, cols], y)
    p_ = m_.predict(F[:, cols])
    r2_ = r2_score(y, p_)
    adj_ = 1 - (1 - r2_) * (len(y) - 1) / (len(y) - len(cols) - 1)
    return np.sqrt(mean_squared_error(y, p_)), r2_, adj_

for label, cols in [("first 7 features", list(range(7))), ("all 9 features   ", list(range(9)))]:
    r_, r2_, a_ = fit_report(cols)
    print(f"{label}: RMSE {r_:.1f} | R2 {r2_:.3f} | adjusted R2 {a_:.3f}")

### E4. Debug Challenge: Find and Fix the Bug
**Scenario:** You want to predict the productivity score of a prospective candidate using the 9-feature `model` trained in E1. Running `.predict()` throws a dimension mismatch error.

> `ValueError: X has 7 features, but LinearRegression is expecting 9 features as input.`

**Your Task:**
1. Run the cell below to see the shape mismatch error.
1. Check `names` to identify which 2 features are missing from `candidate`.
1. Add realistic values for `meeting_hours` (e.g. 14) and `blocker_hours` (e.g. 6) to make it a 9-element array and get a valid prediction.

<details>
<summary>💡 Hint</summary>
Check the <code>names</code> list: the 9 features end with <code>meeting_hours</code> and <code>blocker_hours</code>. <code>candidate</code> only has 7 numbers. Add values for the 8th and 9th features so the shape is <code>(1, 9)</code>.
</details>

In [ ]:
# ❌ BUGGY CODE: Run this cell to observe the error, then fix it!
# Goal: Predict productivity for a new candidate using the 9-feature model.
# Feature order: ['experience_years', 'training_hours', 'task_complexity', 'ai_assistant',
#                 'rework_tickets', 'absent_days', 'team_size', 'meeting_hours', 'blocker_hours']

# Candidate provided with only 7 features:
candidate = np.array([[4, 10, 3, 1, 2, 1, 8]])  # 💥 Missing meeting_hours and blocker_hours!

predicted_score = model.predict(candidate)
print("Predicted productivity score:", round(predicted_score[0], 1))

---
# Part F - Which features affect productivity? (the main idea)
### F1. Rank the features fairly: standardised weights
Raw weights are in different units (years, hours, yes/no), so we cannot compare them. We standardise each feature first (scikit-learn `StandardScaler`), then fit.

In [ ]:
sm = make_pipeline(StandardScaler(), LinearRegression()).fit(F, y)
std_w = sm[-1].coef_                       # points of score per ONE standard deviation of the feature
raw_w = model.coef_

ro = np.argsort(-np.abs(std_w))
print(f"{'rank':4s} {'feature':18s} {'raw weight':>10s} {'1 std dev':>9s} {'std weight':>10s}")
for r_, j in enumerate(ro, 1):
    print(f"{r_:<4d} {names[j]:18s} {raw_w[j]:+10.2f} {sd[j]:9.2f} {std_w[j]:+10.2f}")

plt.figure(figsize=(6, 3.4))
colors = ["seagreen" if v > 0 else "indianred" for v in std_w[ro][::-1]]
plt.barh([names[j] for j in ro][::-1], std_w[ro][::-1], color=colors)
plt.axvline(0, color="black", lw=0.8); plt.title("Standardised weights (points per typical step)")
plt.tight_layout(); plt.show()
print("Same as raw weight x std dev:", np.allclose(std_w, raw_w * sd))

### F2. The drop-one test: remove a feature, refit, see what we lose

In [ ]:
r2_full = r2_score(y, model.predict(F))
print("R2 with all nine features:", round(r2_full, 3))
print(f"{'feature removed':18s} {'R2 without':>10s} {'loss':>7s} {'R2 alone':>9s}")
drop = {}
for j in range(9):
    keep = [k_ for k_ in range(9) if k_ != j]
    r2_k = r2_score(y, LinearRegression().fit(F[:, keep], y).predict(F[:, keep]))
    drop[j] = r2_full - r2_k
for j in sorted(drop, key=lambda j: -drop[j]):
    alone = cors[j] ** 2
    print(f"{names[j]:18s} {r2_full - drop[j]:10.3f} {drop[j]:7.3f} {alone:9.3f}")

### F3. The luck check: bootstrap (300 re-draws, seed 42)

In [ ]:
rng = np.random.default_rng(42)
W = np.zeros((300, 9))
for b_ in range(300):
    i = rng.integers(0, len(y), len(y))                  # random rows, with repeats
    W[b_] = LinearRegression().fit(F[i], y[i]).coef_
low, high = np.percentile(W, 2.5, axis=0), np.percentile(W, 97.5, axis=0)

for j in range(9):
    verdict = "cannot tell (range includes 0)" if low[j] < 0 < high[j] else "real (one side of 0)"
    print(f"{names[j]:18s} {raw_w[j]:+6.2f}   {low[j]:+6.2f} to {high[j]:+6.2f}   {verdict}")

### F4. What-if questions

In [ ]:
typical = np.array([[4, 8, 3, 0, 3, 1, 8, 16, 10]])     # 4 yrs, 8 h training, complexity 3, no AI, 3 rework, 1 absent, team 8, 16 meeting h, 10 blocker h
start = model.predict(typical)[0]
print("Typical employee, predicted score:", start.round(1))

changes = [("training_hours 8 -> 13", 1, 13), ("ai_assistant 0 -> 1", 3, 1), ("rework_tickets 3 -> 1", 4, 1),
           ("absent_days 1 -> 3", 5, 3), ("task_complexity 3 -> 4", 2, 4), ("meeting_hours 16 -> 10", 7, 10),
           ("blocker_hours 10 -> 5", 8, 5), ("team_size 8 -> 12", 6, 12)]
for label, j, v in changes:
    changed = typical.copy(); changed[0, j] = v
    print(f"{label:26s} {model.predict(changed)[0] - start:+.1f}")

> **A link is not a cause.** The weights describe links in *this* data. A pilot (try the change with one team and compare) is needed before saying a feature *causes* a change.

### F5. Debug Challenge: Find and Fix the Bug
**Scenario:** To compare feature importance, a student attempts to convert raw weights to standardised weights manually: $\text{std\_weight} = \text{raw\_weight} \times \sigma$. However, their manual calculation fails the `np.allclose()` verification against the pipeline's `std_w`.

> The manual weights do NOT match the standardised pipeline weights!

**Your Task:**
1. Run the cell to inspect the comparison table.
1. Identify what property of `scaler` represents standard deviation $\sigma$ (spread) rather than the average $\mu$.
1. Fix the calculation so `matches` evaluates to `True`.

<details>
<summary>💡 Hint</summary>
<code>scaler.mean_</code> is the mean (average), not the standard deviation! In scikit-learn's <code>StandardScaler</code>, the standard deviation is stored in <code>scaler.scale_</code> (or the variable <code>sd</code> from E2).
</details>

In [ ]:
# ❌ BUGGY CODE: Run this cell to observe the error, then fix it!
# Goal: Compute standardised weights manually by multiplying raw weight * standard deviation.
# Student mistakenly used scaler.mean_ instead of the standard deviation!

manual_std_weights = model.coef_ * scaler.mean_  # 💥 Mean is NOT standard deviation!

# Verification against pipeline std_w:
matches = np.allclose(manual_std_weights, std_w)
print("Do manual weights match pipeline std_w?:", matches)
print(f"{'Feature':18s} {'Manual':>10s} {'Actual std_w':>12s}")
for nme, mw, sw in zip(names, manual_std_weights, std_w):
    print(f"{nme:18s} {mw:10.2f} {sw:12.2f}")

---
# Part G - Can we trust the model?
### G1. Residuals

In [ ]:
res = y - pm                          # real minus predicted
print("mean residual      :", res.mean().round(3))
print("share within 2 std :", np.mean(np.abs(res) < 2 * res.std()).round(3))
lowp = pm < np.median(pm)
print("spread low vs high :", res[lowp].std().round(1), res[~lowp].std().round(1))
z = np.abs(res) / res.std()
print("worst z-score:", z.max().round(1), "| records beyond 3 std:", int(np.sum(z > 3)))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].scatter(pm[sample], res[sample], s=6, alpha=0.4); ax[0].axhline(0, color="red")
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("residual"); ax[0].set_title("Residuals vs predictions (should be a shapeless band)")
ax[1].hist(res, bins=40, color="steelblue"); ax[1].set_title("Residuals (should be bell-shaped)")
plt.tight_layout(); plt.show()

### G2. Does the model drift over time? (the AI assistant arrives in month 13)

In [ ]:
print("months   records  mean residual  share using AI")
for lo_, hi_ in [(1, 6), (7, 12), (13, 18), (19, 24)]:
    era = (mo >= lo_) & (mo <= hi_)
    print(f"{lo_:>2d}-{hi_:<2d}   {era.sum():7d}  {res[era].mean():+13.2f}  {F[era, 3].mean():13.0%}")

### G3. Features that copy each other: VIF

In [ ]:
# VIF = 1 / (1 - R2), where R2 comes from predicting one feature from the other eight
vif = []
for j in range(9):
    others = [k_ for k_ in range(9) if k_ != j]
    r2_j = LinearRegression().fit(F[:, others], F[:, j]).score(F[:, others], F[:, j])
    vif.append(1 / (1 - r2_j))
for nme, v_ in zip(names, vif):
    print(f"{nme:18s} {v_:.2f}")
print("All VIF values far below 5 -> the features do not copy each other.")

# The extreme case: add experience in months (an exact copy of experience_years x 12)
F_dup = np.column_stack([F, F[:, 0] * 12])
dup = LinearRegression().fit(F_dup, y)
print("columns:", F_dup.shape[1], "but independent columns found by sklearn (rank_):", dup.rank_)   # 10 vs 9
print("years weight + 12 x months weight =", round(dup.coef_[0] + 12 * dup.coef_[-1], 2),
      "(only this combination is meaningful; it equals the experience weight", round(model.coef_[0], 2), ")")

### G4. Debug Challenge: Find and Fix the Bug
**Scenario:** A student attempts to compute the Variance Inflation Factor (VIF) for feature 0 (`experience_years`). However, their calculated VIF is unexpectedly high (~3.87) because they mistakenly regressed against the target label `y` instead of feature 0.

> VIF is an unsupervised diagnostic of multicollinearity among FEATURES. Target `y` must never be used!

**Your Task:**
1. Run the cell below to see the incorrect calculation.
1. Recall that VIF for feature $j$ comes from regressing feature $j$ against all other features ($X_j$ ~ $X_{-j}$).
1. Fix the target in `.fit()` and `.score()` from `y` to `F[:, 0]`, and verify that the correct VIF is ~1.08.

<details>
<summary>💡 Hint</summary>
Replace <code>y</code> with <code>F[:, 0]</code> in both <code>LinearRegression().fit(F[:, other_features], ...)</code> and <code>.score(F[:, other_features], ...)</code>.
</details>

In [ ]:
# ❌ BUGGY CODE: Run this cell to observe the error, then fix it!
# Goal: Compute VIF for feature 0 (experience_years).
# VIF = 1 / (1 - R^2), where R^2 comes from regressing feature 0 against the other 8 features.

other_features = [1, 2, 3, 4, 5, 6, 7, 8]

# 💥 Bug: Regressing against target 'y' (productivity score) instead of feature 0 (F[:, 0])!
vif_model = LinearRegression().fit(F[:, other_features], y)
r2_wrong = vif_model.score(F[:, other_features], y)
vif_wrong = 1 / (1 - r2_wrong)

print("Student's calculated VIF:", round(vif_wrong, 2))
print("Expected VIF for experience_years: ~1.08 (features do not copy each other)")

---
# Part H - Working on new data
### H1. Random 80/20 split with `train_test_split`

In [ ]:
F_tr, F_te, y_tr, y_te = train_test_split(F, y, test_size=0.2, random_state=42)
m_tr = LinearRegression().fit(F_tr, y_tr)
print("train RMSE:", np.sqrt(mean_squared_error(y_tr, m_tr.predict(F_tr))).round(2))
print("test  RMSE:", np.sqrt(mean_squared_error(y_te, m_tr.predict(F_te))).round(2))
# The 5 folds follow the row order (which is ordered by month), so the scores differ a little: the AI assistant only appears from month 13
print("5-fold cross-validated R2:", cross_val_score(LinearRegression(), F, y, cv=5, scoring="r2").round(3))

### H2. A split by time: the honest test for predicting the future

In [ ]:
past, future = mo <= 18, mo >= 19          # 4,385 and 1,453 records
m_t = LinearRegression().fit(F[past], y[past])
pf = m_t.predict(F[future])
print("train RMSE:", np.sqrt(mean_squared_error(y[past], m_t.predict(F[past]))).round(2))
print("test  RMSE:", np.sqrt(mean_squared_error(y[future], pf)).round(2))
print("average error on the future months:", np.mean(y[future] - pf).round(2))
base_t = DummyRegressor(strategy="mean").fit(F[past], y[past])
print("baseline RMSE (average of the past):", round(rmse(y[future], base_t.predict(F[future])), 1))   # about 10.4

Split the way you will use the model. A random split quietly lets the model peek at the future.

### H3. Too simple and too complex: underfitting and overfitting

In [ ]:
rng2 = np.random.default_rng(5)
small = rng2.permutation(len(y))[:30]                  # only 30 records to train on
rest = np.setdiff1d(np.arange(len(y)), small)
xe = F[:, [0]]                                         # experience only

print("degree  train RMSE  test RMSE")
for deg in [0, 1, 2, 5]:
    poly = make_pipeline(StandardScaler(), PolynomialFeatures(deg), LinearRegression())
    poly.fit(xe[small], y[small])
    print(f"{deg:>6d}  {rmse(y[small], poly.predict(xe[small])):10.1f}  {rmse(y[rest], poly.predict(xe[rest])):9.1f}")

### H4. Ridge regression: keeping the weights small

In [ ]:
# Ridge on standardised features. alpha is the penalty strength (lambda in the guide).
print("weights per standard deviation (order:", ", ".join(n_.split("_")[0] for n_ in names), ")")
for lam in [0, 1000, 10000]:
    rd = make_pipeline(StandardScaler(), Ridge(alpha=lam)).fit(F, y)
    print(f"alpha={lam:<6} intercept {rd[-1].intercept_:.2f}  weights {rd[-1].coef_.round(2)}")

# Choose the strength by cross-validation instead of guessing
rcv = make_pipeline(StandardScaler(), RidgeCV(alphas=[0.1, 1, 10, 100, 1000, 10000])).fit(F, y)
print("Best alpha chosen by RidgeCV:", rcv[-1].alpha_)

### H5. Debug Challenge: Find and Fix the Bug
**Scenario:** After performing an 80/20 train/test split, a student attempts to evaluate the test set RMSE. Running the evaluation throws an inconsistent sample count error.

> `ValueError: Found input variables with inconsistent numbers of samples: [1168, 4670]`

**Your Task:**
1. Run the cell below to witness the sample mismatch error.
1. Check the dimensions: `y_test` has 1,168 rows, but what was passed to `.predict()`?
1. Fix the feature matrix passed into `eval_model.predict()` to evaluate on the test split (`F_test`).

<details>
<summary>💡 Hint</summary>
The code currently calls <code>eval_model.predict(F_train)</code> while comparing against <code>y_test</code>! Change <code>F_train</code> to <code>F_test</code>.
</details>

In [ ]:
# ❌ BUGGY CODE: Run this cell to observe the error, then fix it!
# Goal: Evaluate model performance honestly on the test set.
# Split: 80% train, 20% test.

F_train, F_test, y_train, y_test = train_test_split(F, y, test_size=0.2, random_state=42)
eval_model = LinearRegression().fit(F_train, y_train)

# 💥 Bug: Comparing y_test (1,168 samples) with predictions on F_train (4,670 samples)!
test_rmse = np.sqrt(mean_squared_error(y_test, eval_model.predict(F_train)))
print("Honest Test RMSE:", round(test_rmse, 2))

---
# Wrap-up
* **NumPy** gives fast arrays, masks, broadcasting and matrix maths. We used it to hold, clean and explore the data.
* **scikit-learn** built every regression model in this notebook: `LinearRegression`, `SGDRegressor` (gradient descent), `StandardScaler`, `Ridge`/`RidgeCV`, `train_test_split`, cross-validation and the metrics.
* **Findings (simulated data):** experience alone explains about 23% of the ups and downs in the score; all nine features together explain about 75%. Team size adds nothing. The strongest effects are experience, task complexity, absent days, rework and the AI assistant; blocker hours, training hours and meeting hours add smaller but real effects.
* **Use responsibly:** use this to understand what helps people do well (training, tools, fewer blockers, task planning). Do **not** use it to rank people or decide pay. About a quarter of the score is still unexplained, and features such as absent days or rework may reflect someone's situation, not their effort.

### Practice (after the session)
1. Repeat Part C for `blocker_hours` and `meeting_hours` and compare the slopes with the nine-feature weights.
2. Do the drop-one test for the **pair** `meeting_hours` + `blocker_hours` together. Do they share information?
3. Use `RidgeCV` with a time split instead of a random split. Does the best `alpha` change?
4. Why should we not trust the model for a person with 25 years of experience (the file has none above 15)?